# BigQuery Time Travel & Change History — Runnable Demo

In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import time

PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"
DATASET = "orders"
TABLE = "inventory_demo"
TABLE_ID = f"{PROJECT_ID}.{DATASET}.{TABLE}"

client = bigquery.Client(project=PROJECT_ID)

def run(sql):
    return client.query(sql).result().to_dataframe()


In [ ]:
run(f"""
CREATE OR REPLACE TABLE `{TABLE_ID}`
(
  product_id INT64,
  product_name STRING,
  quantity INT64
)
OPTIONS (
  enable_change_history = TRUE
)
""")
print("table created")


In [ ]:
T0 = run("SELECT CURRENT_TIMESTAMP() AS t")["t"][0]
print("T0 =", T0)


In [ ]:
run(f"""
INSERT INTO `{TABLE_ID}` (product_id, product_name, quantity)
VALUES (1, 'Widget', 100), (2, 'Gadget', 50), (3, 'Gizmo', 75)
""")
print("tranche 1 inserted")


In [ ]:
run(f"""
select * from `{TABLE_ID}`
""")

In [ ]:
time.sleep(2)
T1 = run("SELECT CURRENT_TIMESTAMP() AS t")["t"][0]
print("T1 =", T1)


In [ ]:
run(f"""
UPDATE `{TABLE_ID}` SET quantity = quantity - 20 WHERE product_id = 1
""")
run(f"""
INSERT INTO `{TABLE_ID}` (product_id, product_name, quantity)
VALUES (4, 'Doohickey', 30)
""")
run(f"""
DELETE FROM `{TABLE_ID}` WHERE product_id = 2
""")
print("tranche 2 applied: update + insert + delete")


In [ ]:
run(f"""
select * from `{TABLE_ID}`
""")

In [ ]:
time.sleep(2)
T2 = run("SELECT CURRENT_TIMESTAMP() AS t")["t"][0]
print("T2 =", T2)


In [ ]:
print("Current state:")
display(run(f"SELECT * FROM `{TABLE_ID}` ORDER BY product_id"))

print("Time travel -> state as of T1:")
display(run(f"""
SELECT * FROM `{TABLE_ID}`
FOR SYSTEM_TIME AS OF TIMESTAMP('{T1}')
ORDER BY product_id
"""))

print("Time travel -> state as of T0:")
display(run(f"""
SELECT * FROM `{TABLE_ID}`
FOR SYSTEM_TIME AS OF TIMESTAMP('{T0}')
ORDER BY product_id
"""))


In [ ]:
# Cleanup -- uncomment to drop the demo table
run(f"DROP TABLE `{TABLE_ID}`")
